# Execution Testing

The aim of this notebook is to verify if functions execute correctly on randomly generated data and to check how long they take to execute to guide their use.

## Setup

In [1]:
import os
import time
from contextlib import contextmanager
from pathlib import Path
from typing import get_args

import numpy as np

In [2]:
# Find project root (folder that contains .git)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

# Set working directory to root
os.chdir(ROOT)

print("Now working in:", Path.cwd())

Now working in: C:\Users\couch\OneDrive\Assignments\Master Thesis\Repo


In [3]:
from src.config import WINDOW_SIZE
from src.metrics import MetricName, compute_metric
from src.portfolio_optimization import optimize_portfolio

In [4]:
@contextmanager
def time_block(label: str = "Block"):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"[{label}] Elapsed time: {elapsed:.6f}s")

In [5]:
# Reproducible matrix
rng = np.random.default_rng(seed=42)
matrix = rng.standard_normal((WINDOW_SIZE, 20))

## Metric Testing

In [6]:
for metric in get_args(MetricName):
    with time_block(metric):
        try:
            value = compute_metric(matrix[:, 0], metric=metric)
            print(f"[{metric}] Value: {value:.6f}")
        except Exception as e:
            print(f"[{metric}] Error: {e}")

[sharpe] Value: 0.005977
[sharpe] Elapsed time: 0.000189s
[sortino] Value: 0.008376
[sortino] Elapsed time: 0.000036s
[central_sortino] Value: 0.008338
[central_sortino] Elapsed time: 0.000016s
[omega] Value: 0.015167
[omega] Elapsed time: 0.000013s
[tail_effectiveness] Value: 0.002831
[tail_effectiveness] Elapsed time: 0.000573s
[central_tail_effectiveness] Value: 0.002823
[central_tail_effectiveness] Elapsed time: 0.000086s


All metrics are exceptionally fast and the ones that are slower are so for being lower in the conditional hierarchy.

## GA Testing

In [7]:
def ga_eval_func(ret_mat, w) -> float:
    port_excess = np.sum(ret_mat * w, axis=1)
    return compute_metric(port_excess, metric="tail_effectiveness")


with time_block("GA optimization"):
    try:
        value, pop = optimize_portfolio(
            matrix,
            ga_eval_func,
            ngen=20,
            pop_size=200,
            cxpb=0.7,
            mutpb=0.2,
            max_shift=20,
            indpb=0.5,
            tournsize=2,
            total_tokens=1000,
        )
        print("[GA] Value:")
        print(value)
        print(f"[GA] Weight sum: {sum(value)}")
    except Exception as e:
        print(f"[GA] Error: {e}")

[GA] Value:
[0.028 0.007 0.001 0.065 0.07  0.245 0.004 0.009 0.047 0.086 0.004 0.159
 0.001 0.001 0.084 0.002 0.007 0.14  0.04  0.   ]
[GA] Weight sum: 1.0
[GA optimization] Elapsed time: 0.672554s
